# 07. Reject Invalid Subscriber #2

## 07.1. Confirmation Email

### 7.1.0 Overview

Main reason we prioritize sending our newsletter subscribers a confirmation email, is because we want explicit consent for the
subscribers that we can send them newletter issues. This allows us to;
1. Shield our users from abuse (EU mandates explicit consent from the user for example)
2. We get to confirm that the email addresses they provided actually exist before trying to send them a newsletter

# 7.1.1. The Confirmation User Journey

Every time a user want to subscribe to our newsletter they fire a `POST /subscriptions` request. Our request handler will
- Add their details to our database in the `subscriptions` table, with `statux` equal to `pending_confirmation`
- generate (unique) `subscription_token`;
- store `subscription_token` in our database against their `id` in a `subscription_tokens` table;
- send an email to the new subscriber containing a link structured as `https:://<api-domain>/subscriptions/confirm?token=<subscription_token>`;
- return a `200 OK`

Once a user clicks on the link, a browser tab will open up and `GET` request will be fired to our `GET /subscription/confirm` endpoint. Our request
handler will:
- retrieve `subscription_token` from the query parameters
- retrieve the subscriber `id` associated with the `subscription_token` from the `subscriptions_token` table;
- update the subscriber status from `pending_confirmation` to active in the `subscriptions` table;
- return a `200 OK`.

<div style="background-color: #313B51; color: white; padding: 15px; border-radius: 15px;">

<h4>Note:</h4>
    
There are a few other possible designs/approaches (e.g. use a JWT instead of a unique token) and we have a few edge/corner cases to handle e.g.
- What happens if they click on the link twice?
- What happends if they try to subscribe twice?

We will discuss both at the most appropriate time as we make progress with the implementation.

</div>

The implementation strategy
1. Write a module to send an email
2. adapt the logic of our existing `POST /subscriptions` request handler to match new specifications
3. write a `GET /subscriptions/confirm` request handler from scratch.

## 07.2. EmailClient, Our Email Delivery Component

### 7.2.1. How to Send An Email

#### 7.2.1.0. Overview

How do we _actually_ send an email?

How does it work?

We have to look into **SMTP**
> **S**imple **M**ail **T**ransfer **P**rotocol
>
> Its been around since the early days of the internet - the first RFC dates back to 1982.
>

SMTP does for emails what HTTP does for web pages:
> it is an application-level protocol that ensures that different implementations of email servers and clients can understand each other and
> exchange messages.

We will not be building our own private email server, it will take too long and we wouldn't gain much from the effort. We will be leveraging a
third-party service.

Do we need to talk SMTP to these third party services?

Not necessarily.

SMTP is a specialised protocol. Learning a new protocol takes time and we are bound to make mistakes along the way - that is why most providers
provide two interfaces: SMTP and REST API

If we were familiar with the email protocol, or we needed some non-conventional configuration, we would use the SMTP interface. Otherwise we will
get up and running much faster and more reliably using a REST API

#### 7.2.1.1. Choosing an Email API

For our implementation we will work with 
- [Postmark](https://account.postmarkapp.com/servers) - for following along with the book (need a work email with private domain)
- [MailJet](https://app.mailjet.com/onboarding) - a Postmark alternative (Works with public domains [gmail, yahoo] for now)

#### 7.2.1.2. The Email Client Interface

There are usually two approaches when it comes to a new piece of functionality
1. Bottom-Up - starting from the implementation details and slowly working our way up
2. Top-Down - designing the interface first and then figuring out how the implementation is going to work (to an extent).

In this case, we will go for the second route.

What kind of interface do we want for our email client?

We'd like to have some kind of `send_email` method. At the moment we just need to send a single email out at a time - we will deal
with the complexity of sending emails in batches when we start working on newsletter issues.

What arguments should `send_email` accept?

We will definitely need;
- recipient email address,
- the subject line,
- email contents
    - We'll ask for both HTML and plain versions of the email content
- sender email address? We'll assume that all emails sent by an instance of the client are coming from the same address, therefore we
  do not need it as an argument of `send_email`, it will be one of the arguments in the constructor of the client itself.

We will also expect `send_email` to be an asynchronous function, given that we will be performing I/O to talk to a remote server.

Stiching everything together, we have something that looks more or less like this:

```Rust
//! src/email_client.rs

use crate::domain::SubscriberEmail;

pub struct EmailClient {
    sender: SubscriberEmail;
}

impl EmailClient {
    pub async fn send_email(
        &self,
        _recipient: SubscriberEmail,
        _subject: &str,
        _html_content: &str,
        _text_content: &str,
    ) -> Result<(), String>
    {
        todo!("NOT YET IMPLEMENTED")
        
    }
}


//! src/lib.rs

pub mod configuration;
pub mod domain;
// New entry!
pub mod email_client;
pub mod routes;
pub mod startup;
pub mod telemetry;
```

There is an unresolved question - the return type. We sketched a `Result<(), String>` which is a way to spell _"I'll think about error handling later"_.

### 7.2.2. How To Write A REST Client Using `reqwest`

### 7.2.3. How to Test A REST Client Using `reqwest`

### 7.2.4. First Sketch Of `EmailClient::send_email`

### 7.2.5. Tightening Our Happy Path Test

### 7.2.6. Dealing With Failures

## 7.3. Skeleton And Principles For A Maintainable Test Suite

## 7.4. Refocus

## 7.5. Zero Downtime Deployment

## 7.6. Database Migrations

## 7.7. Sending A Confirmation Email

## 7.8. Database Transactions

## 7.9. Summary